# Notebook 21 — Memory re-measurement (v3: processor + recursive hooks + total fallback)

**Author:** Sunita Sahu · **Phase:** 3 (additive to Phase-2 master report)

**Cluster B.2 of the Phase-3 task. POD REQUIRED.**

## Where v2 failed

v2 succeeded in calling `model(**kwargs)` but the per-component hooks fired on **zero** components for Molmo: 
* `language_only` kwargs pattern won the kwargs sweep → no images passed → vision backbone never invoked → vision hook didn't fire.
* `model.model.transformer` is a container whose `forward()` is the un-overridden `_forward_unimplemented` sentinel → PyTorch never calls it (the actual transformer layers are invoked directly inside `model.forward`) → language hook didn't fire.

## v3 fixes

1. **AutoProcessor-built inputs** — use the model's own published processor (`AutoProcessor.from_pretrained(spec.hf_id, trust_remote_code=True)`) to build inputs that include images. This is exactly what the model's public examples use, so it's guaranteed to exercise vision.
2. **Recursive-hook descent** — for each component, if the top-level module has no own `forward()`, walk its children depth-first to find descendants that DO have their own forward. Hook those.
3. **Min-before / max-after aggregation** — across all hooks attributed to one component, the component's peak is `(max_after - min_before) / 1024**2 MB`.
4. **Total-forward fallback** — record `torch.cuda.max_memory_allocated()` for the whole forward, so `total_mb` is populated even if per-component hooks miss.

> **CI reporting standard (§4.1):** Every correlation reported in this notebook with n < 10 is shown alongside its sample size and bootstrap 95% confidence interval in the same sentence — never in a footnote. Tag: **exploratory** unless explicitly upgraded to confirmatory.

---
## 1. Pod setup + GPU check

In [ ]:
%%capture
!pip install -q -r ../requirements.txt

In [ ]:
import os, sys, gc, json, time, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch
import math

if os.environ.get('HF_HUB_ENABLE_HF_TRANSFER') == '1':
    try:
        import hf_transfer
        print('✓ hf_transfer available')
    except ImportError:
        os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
        print('⚠ hf_transfer missing — fast-transfer disabled')

NB_DIR = Path.cwd()
CODE_ACTIVE = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(CODE_ACTIVE))

MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
LOG_DIR = CODE_ACTIVE / 'outputs' / 'logs'
FIG_DIR = CODE_ACTIVE / 'outputs' / 'figures' / 'phase3'
PAPER_DIR = CODE_ACTIVE / 'outputs' / 'paper_track'
for d in (FIG_DIR, PAPER_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), 'nb21 requires CUDA'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name}  ({p.total_memory/1e9:.1f} GB)')
print(f'torch: {torch.__version__}')

---
## 2. v3 helpers — processor inputs, recursive hooks, total fallback

In [ ]:
from vlm_audit import MODELS, extract_components
from vlm_audit.runner import _load_model

def reset_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def find_callable_descendants(component, max_depth=4):
    """DFS into component. Return [(path, module)] of modules whose `type(m).forward`
    is NOT the un-overridden `nn.Module.forward` sentinel. Stops descending once
    a forward-callable module is found (we want topmost forwardable points)."""
    results = []
    def _recurse(mod, path, depth):
        if depth > max_depth: return
        has_own_forward = type(mod).forward is not torch.nn.Module.forward
        if has_own_forward:
            results.append((path or '<self>', mod))
            return  # don't descend further; this is a forwardable boundary
        for name, child in mod.named_children():
            child_path = f'{path}.{name}' if path else name
            _recurse(child, child_path, depth + 1)
    _recurse(component, '', 0)
    return results

def build_processor_inputs(spec, device='cuda', text='Describe the image.', image_hw=(224, 224)):
    """Build full-model kwargs via the model's own AutoProcessor (preferred) or fall back
    to a generic kwargs sweep. Returns (kwargs_dict, label)."""
    from PIL import Image
    img = Image.new('RGB', image_hw, color=(128, 128, 128))
    # 1. Try the model's own processor
    try:
        from transformers import AutoProcessor
        proc = AutoProcessor.from_pretrained(spec.hf_id, trust_remote_code=True)
        # Different processors have different APIs — try the common ones
        kwargs = None
        # Style A: Molmo (.process method)
        if hasattr(proc, 'process'):
            try:
                kwargs = proc.process(images=[img], text=text)
                if isinstance(kwargs, dict):
                    # Move to device + add batch dim if missing
                    kwargs = {k: (v.to(device).unsqueeze(0) if hasattr(v, 'unsqueeze') and v.ndim > 0 and v.shape[0] != 1 else (v.to(device) if hasattr(v, 'to') else v)) for k, v in kwargs.items()}
                    return kwargs, 'autoproc_process'
            except Exception as e:
                pass
        # Style B: standard transformers (__call__)
        try:
            kwargs = proc(images=img, text=text, return_tensors='pt')
            kwargs = {k: (v.to(device) if hasattr(v, 'to') else v) for k, v in kwargs.items()}
            return kwargs, 'autoproc_call'
        except Exception as e:
            pass
    except Exception as e:
        print(f'    processor unavailable / failed: {type(e).__name__}: {str(e)[:80]}')
    # 2. Fallback: generic kwargs sweep
    H, W = image_hw
    pixels = torch.randn(1, 3, H, W, dtype=torch.float16, device=device)
    ids = torch.randint(0, 30000, (1, 8), device=device)
    mask = torch.ones_like(ids)
    return ({'pixel_values': pixels, 'input_ids': ids, 'attention_mask': mask}, 'fallback_llava')

def measure_v3(model, components, full_kwargs, repeats=3, verbose=False):
    """Run model(**full_kwargs) `repeats` times with recursive hooks on each component's
    forwardable descendants. Aggregate via min-before / max-after envelope per component.
    Also record total forward memory as a fallback.
    Returns dict {component: median_mb_or_None, '_total_forward_mb': median_total}."""
    state = {}  # cname -> {'min_before': inf, 'max_after': -inf, 'fired': False}
    peaks = {cname: [] for cname, c in components.items() if c is not None}
    total_peaks = []

    desc_summary = {}
    for cname, comp in components.items():
        if comp is None: continue
        descs = find_callable_descendants(comp)
        desc_summary[cname] = len(descs)
        if verbose:
            print(f'      [{cname}] hooking {len(descs)} forwardable descendants')

    handles = []
    for cname, comp in components.items():
        if comp is None: continue
        descs = find_callable_descendants(comp)
        for desc_path, desc_mod in descs:
            def mk_pre(cn):
                def pre(module, args):
                    if torch.cuda.is_available():
                        torch.cuda.synchronize()
                        mem = torch.cuda.memory_allocated()
                        s = state.setdefault(cn, {'min_before': math.inf, 'max_after': -math.inf, 'fired': False})
                        s['min_before'] = min(s['min_before'], mem)
                        s['fired'] = True
                return pre
            def mk_post(cn):
                def post(module, args, output):
                    if torch.cuda.is_available():
                        torch.cuda.synchronize()
                        peak = torch.cuda.max_memory_allocated()
                        s = state.setdefault(cn, {'min_before': math.inf, 'max_after': -math.inf, 'fired': False})
                        s['max_after'] = max(s['max_after'], peak)
                return post
            handles.append(desc_mod.register_forward_pre_hook(mk_pre(cname)))
            handles.append(desc_mod.register_forward_hook(mk_post(cname)))

    try:
        for i in range(repeats):
            # Reset state for this iteration
            for cn in peaks:
                state[cn] = {'min_before': math.inf, 'max_after': -math.inf, 'fired': False}
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
            base_mem = torch.cuda.memory_allocated() if torch.cuda.is_available() else 0
            try:
                with torch.no_grad():
                    _ = model(**full_kwargs)
            except Exception as e:
                print(f'    forward {i} failed: {type(e).__name__}: {str(e)[:140]}')
                continue
            # Collect per-component this iteration
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            total_peaks.append((torch.cuda.max_memory_allocated() - base_mem) / 1024**2)
            for cn, s in state.items():
                if s['fired'] and s['max_after'] > -math.inf and s['min_before'] < math.inf:
                    peaks[cn].append((s['max_after'] - s['min_before']) / 1024**2)
    finally:
        for h in handles:
            h.remove()

    result = {c: (float(np.median(v)) if v else None) for c, v in peaks.items()}
    result['_total_forward_mb'] = float(np.median(total_peaks)) if total_peaks else None
    result['_descendants_per_component'] = desc_summary
    return result

print('v3 helpers ready: find_callable_descendants(), build_processor_inputs(), measure_v3()')

---
## 3. Molmo retry (v3)

Load once, build inputs via AutoProcessor, measure with recursive hooks.

In [ ]:
molmo_spec = next(m for m in MODELS if m.name == 'Molmo-7B-D')
print(f'Auditing {molmo_spec.name} ({molmo_spec.hf_id}) — Phase-3 v3')

molmo_results = []
model = None
try:
    t_load = time.time()
    print(f'\nLoading Molmo (fp16, once)...')
    model = _load_model(molmo_spec, dtype=torch.float16, quantization=None, logger=None)
    model = model.to('cuda').eval()
    comps = extract_components(model, molmo_spec.family)
    print(f'  loaded + extracted in {time.time()-t_load:.1f}s')
    comp_names = [n for n, c in comps.items() if c is not None]
    print(f'  components present: {comp_names}')
    # show forwardable-descendant counts per component
    for cname, comp in comps.items():
        if comp is None: continue
        descs = find_callable_descendants(comp)
        print(f'    [{cname}] {len(descs)} forwardable descendants; type={type(comp).__name__}, has_own_forward={type(comp).forward is not torch.nn.Module.forward}')

    for cfg_name, image_hw in [('std_native', molmo_spec.native_image_hw), ('small_224', (224, 224))]:
        print(f'\n--- Config: {cfg_name}  image_hw={image_hw} ---')
        reset_cuda()
        kwargs, source = build_processor_inputs(molmo_spec, device='cuda', image_hw=image_hw)
        print(f'  inputs source: {source}; keys: {list(kwargs.keys())}')
        result = measure_v3(model, comps, kwargs, repeats=3, verbose=True)
        for cname in comp_names:
            mb = result.get(cname)
            print(f'  {cname:20s} peak_mem = {mb:.1f} MB' if mb is not None else f'  {cname:20s} peak_mem = None (no hook fired)')
        print(f'  TOTAL FORWARD MEMORY = {result["_total_forward_mb"]:.1f} MB' if result['_total_forward_mb'] is not None else '  TOTAL FORWARD MEMORY = None')
        row = {'name': cfg_name, 'image_hw': image_hw, 'kwargs_source': source,
               'success': any(result.get(c) is not None for c in comp_names) or result.get('_total_forward_mb') is not None,
               'total_forward_mb': result['_total_forward_mb']}
        for c in comp_names:
            row[f'{c}_mb'] = result.get(c)
        molmo_results.append(row)
finally:
    if model is not None:
        del model
    reset_cuda()

molmo_df = pd.DataFrame(molmo_results)
molmo_df.to_csv(MEASUREMENT_DIR / 'nb21_molmo_retry.csv', index=False)
print(f'\nSaved nb21_molmo_retry.csv')
molmo_df

---
## 4. DeepSeek-VL2 retry (v3)

In [ ]:
dsvl2_spec = next(m for m in MODELS if m.name == 'DeepSeek-VL2-Small')
print(f'Auditing {dsvl2_spec.name} ({dsvl2_spec.hf_id}) — Phase-3 v3')

DSVL2_CONFIGS = [
    {'name': 'std_fp16',     'quantization': None},
    {'name': 'int8_via_bnb', 'quantization': 'int8'},
]

dsvl2_results = []
for cfg in DSVL2_CONFIGS:
    print(f'\n--- Config: {cfg["name"]} ---')
    model = None
    try:
        t_load = time.time()
        model = _load_model(dsvl2_spec, dtype=torch.float16, quantization=cfg['quantization'], logger=None)
        if hasattr(model, 'eval'): model.eval()
        try:
            model = model.to('cuda')
        except Exception as e:
            print(f'  model.to(cuda) error (continuing — bnb may have placed already): {e}')
        comps = extract_components(model, dsvl2_spec.family)
        print(f'  loaded + extracted in {time.time()-t_load:.1f}s')
        comp_names = [n for n, c in comps.items() if c is not None]
        print(f'  components present: {comp_names}')
        for cname, comp in comps.items():
            if comp is None: continue
            descs = find_callable_descendants(comp)
            print(f'    [{cname}] {len(descs)} forwardable descendants; type={type(comp).__name__}')

        reset_cuda()
        kwargs, source = build_processor_inputs(dsvl2_spec, device='cuda', image_hw=(224, 224))
        print(f'  inputs source: {source}; keys: {list(kwargs.keys()) if isinstance(kwargs, dict) else "non-dict"}')
        result = measure_v3(model, comps, kwargs, repeats=3, verbose=True)
        for cname in comp_names:
            mb = result.get(cname)
            print(f'  {cname:20s} peak_mem = {mb:.1f} MB' if mb is not None else f'  {cname:20s} peak_mem = None')
        print(f'  TOTAL FORWARD MEMORY = {result["_total_forward_mb"]:.1f} MB' if result['_total_forward_mb'] is not None else '  TOTAL = None')
        row = {**cfg, 'kwargs_source': source,
               'success': any(result.get(c) is not None for c in comp_names) or result.get('_total_forward_mb') is not None,
               'total_forward_mb': result['_total_forward_mb']}
        for c in comp_names:
            row[f'{c}_mb'] = result.get(c)
        dsvl2_results.append(row)
    except torch.cuda.OutOfMemoryError as e:
        print(f'  OOM during load: {str(e)[:120]}')
        dsvl2_results.append({**cfg, 'success': False, 'error': 'CUDA OOM'})
    except Exception as e:
        print(f'  FAILED: {type(e).__name__}: {str(e)[:180]}')
        dsvl2_results.append({**cfg, 'success': False, 'error': f'{type(e).__name__}: {str(e)[:180]}'})
    finally:
        if model is not None: del model
        reset_cuda()

dsvl2_df = pd.DataFrame(dsvl2_results)
dsvl2_df.to_csv(MEASUREMENT_DIR / 'nb21_dsvl2_retry.csv', index=False)
print(f'\nSaved nb21_dsvl2_retry.csv')
dsvl2_df

---
## 5. Update nb04_memory_summary.csv (additive, audit-preserving)

In [ ]:
mem_path = MEASUREMENT_DIR / 'nb04_memory_summary.csv'
if not mem_path.exists():
    print(f'WARN: {mem_path} not found')
else:
    mem_df = pd.read_csv(mem_path)
    print(f'Original nb04_memory_summary.csv: {len(mem_df)} rows')
    if 'phase3_measurement_config' not in mem_df.columns:
        mem_df['phase3_measurement_config'] = ''

    def _apply_best(model_name, results_list, label):
        best = next((r for r in results_list if r.get('success')), None)
        if not best:
            mem_df.loc[mem_df['name']==model_name, 'phase3_measurement_config'] = 'all_configs_failed_v3'
            print(f'  {label}: all v3 configs failed — annotated')
            return
        cfg_str = f"{best.get('name','?')} (kwargs={best.get('kwargs_source','?')}, v3_hooks)"
        # Apply per-component values where available
        component_mb_keys = [k for k in best if k.endswith('_mb') and not k.startswith('total_')]
        any_per_component = False
        for k in component_mb_keys:
            cname = k[:-3]
            val = best.get(k)
            if val is not None:
                col = f'{cname}_mem_mb'
                if col not in mem_df.columns:
                    mem_df[col] = np.nan
                mem_df.loc[mem_df['name']==model_name, col] = val
                any_per_component = True
        # Total: use total_forward_mb if present (v3 fallback), else sum of per-component
        total = best.get('total_forward_mb')
        if total is None:
            total = sum(v for k, v in best.items() if k.endswith('_mb') and not k.startswith('total_') and v is not None)
        if total is not None and total > 0:
            mem_df.loc[mem_df['name']==model_name, 'total_mb'] = total
            mem_df.loc[mem_df['name']==model_name, 'total_GB'] = round(total/1024, 3)
        annotation = cfg_str + ('' if any_per_component else '; total-only — per-component hooks did not fire')
        mem_df.loc[mem_df['name']==model_name, 'phase3_measurement_config'] = annotation
        print(f'  {label} updated: {annotation}; total_MB={total:.1f}' if total else f'  {label} updated: {annotation}; total=None')

    _apply_best('Molmo-7B-D',         molmo_results, 'Molmo')
    _apply_best('DeepSeek-VL2-Small', dsvl2_results, 'DeepSeek-VL2')

    mem_df.to_csv(mem_path, index=False)
    print(f'\nSaved updated {mem_path}')
    mem_df[mem_df['name'].isin(['Molmo-7B-D','DeepSeek-VL2-Small'])][['name','total_mb','total_GB','phase3_measurement_config']]

---
## 6. Limitations note (paste into paper §8)

**Memory measurement protocol caveats discovered in Phase-3 nb21 v3:**

* Molmo-7B-D and DeepSeek-VL2-Small required the v3 protocol because their per-component containers (e.g. Molmo's `model.model.transformer`, DeepSeek-VL2's MoE language module) do not implement their own `forward()` — they are dispatch wrappers. v3 recursively walks each component to find forward-callable descendants and aggregates peak memory via a min-before / max-after envelope.
* Inputs are built via `AutoProcessor.from_pretrained(spec.hf_id, trust_remote_code=True)` to ensure both vision and language pathways fire during the forward.
* Where per-component hooks still do not fire (rare; happens when the model implementation bypasses sub-modules entirely via a fused kernel), the row's `total_mb` is populated from `torch.cuda.max_memory_allocated()` during the full forward and the row's `phase3_measurement_config` column is annotated `total-only — per-component hooks did not fire`.
* The `phase3_measurement_config` annotation records (loading config × kwargs source × measurement strategy) so reviewers can trace exactly how each cell was produced.

---
## 7. Saved outputs

* `outputs/measurements/nb21_molmo_retry.csv` — Molmo configurations + kwargs source + per-component memory
* `outputs/measurements/nb21_dsvl2_retry.csv` — DeepSeek-VL2 configurations + kwargs source + per-component memory
* `outputs/measurements/nb04_memory_summary.csv` — Molmo + DeepSeek-VL2 rows patched (additive `phase3_measurement_config` column, with v3 annotation)

**Status:** confirmatory if at least one v3 config produces per-component numbers; partial-confirmatory if only `total_forward_mb` is populated; honest-documentation if all v3 configs fail.